In [4]:
import json
import pandas as pd

with open("mental.json", "r", encoding="utf-8") as df:
    data = json.load(df)

tables = {}

for item in data:
    if item.get("type") == "table":
        table_name = item["name"]
        tables[table_name] = pd.DataFrame(item["data"])

print(tables.keys())

dict_keys(['highlight_phq', 'highlight_range', 'post', 'post_tag', 'report_post', 'tag', 'user'])


In [5]:
post = tables["post"]
post_tag = tables["post_tag"]
user = tables["user"]
tag = tables["tag"]
report_post = tables["report_post"]

In [6]:
df = (
    post
    .merge(post_tag, left_on="id", right_on="post_id")
    .merge(tag, left_on="tag_id", right_on="id")
)

df = df[["txtContent", "tagName"]]

df.head()

,txtContent,tagName
0,None,ไม่แน่ใจ
1,None,ไม่แน่ใจ
2,None,ไม่แน่ใจ
3,None,ไม่แน่ใจ
4,None,ปกติ


In [7]:
df = df.dropna(subset=["txtContent"])
df = df[df["txtContent"].str.strip() != ""]
df = df.drop_duplicates(subset=["txtContent"])

In [8]:
import re

def clean_text(text):

    text = str(text)

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"www\S+", "", text)

    text = re.sub(r"@\w+", "", text)

    text = re.sub(r"#\w+", "", text)

    text = re.sub(r"\n", " ", text)

    text = re.sub(r"\s+", " ", text)

    text = re.sub(r"[^\u0E00-\u0E7Fa-zA-Z0-9 ]", "", text)

    return text.strip()

df["clean_text"] = df["txtContent"].apply(clean_text)

In [9]:
from pythainlp.tokenize import word_tokenize

df["tokens"] = df["clean_text"].apply(
    lambda x: word_tokenize(x, engine="newmm")
)

from pythainlp.corpus.common import thai_stopwords

stopwords = set(thai_stopwords())

df["tokens"] = df["tokens"].apply(
    lambda words: [w for w in words if w not in stopwords]
)

df["processed_text"] = df["tokens"].apply(
    lambda x: " ".join(x)
)

In [10]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["label"] = le.fit_transform(df["tagName"])

In [11]:
from sklearn.model_selection import train_test_split

X = df["txtContent"]
y = df["tagName"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

X = tfidf.fit_transform(df["processed_text"])

y = df["label"]

print(X)
print(y)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 80944 stored elements and shape (2100, 5000)>
  Coords	Values
  (0, 2836)	1.0
  (1, 4236)	0.13727913603371972
  (1, 2173)	0.056436618970552924
  (1, 1593)	0.09393441556302035
  (1, 398)	0.1072674975738475
  (1, 521)	0.14585599045684747
  (1, 4523)	0.08465804592882333
  (1, 4013)	0.058868776496278835
  (1, 1546)	0.1426706016395178
  (1, 3458)	0.20697338640733273
  (1, 577)	0.20525203927546734
  (1, 6)	0.1143233836586011
  (1, 4819)	0.07605950572425656
  (1, 4100)	0.15777751668950118
  (1, 2099)	0.15098696060010386
  (1, 2946)	0.08576678520314962
  (1, 1887)	0.23634566433309326
  (1, 2918)	0.07484921526904297
  (1, 592)	0.13421280943874275
  (1, 3582)	0.11670648780830964
  (1, 216)	0.057532720926802984
  (1, 1764)	0.07210210306255263
  (1, 2939)	0.0997058888334852
  (1, 4816)	0.14415752232881357
  (1, 4257)	0.06003231544448029
  :	:
  (2098, 3114)	0.10862244814541408
  (2098, 4181)	0.14895984766750278
  (2098, 4918)	0.30840069

In [13]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

model.fit(X_train, y_train)

ValueError: could not convert string to float: '31 ธันวาคม 2567\r\n\r\nก็วันนี้เป็นวันสุดท้ายของปี ก็เลยตัดสินใจทำก๋วยเตี๋ยวกับครอบครัว เลยตื่นขึ้นเร็วหน่อย ประมาณ 6 โมงเช้า เตรียมน้ำก๋วยเตี๋ยวแล้วก็ ต้มก๋วยเตี๋ยวทิ้งไว้ ประมาณ 9 โมงก็เสร็จเรียบร้อย เลยได้กินก๋วยเตี๋ยวเป็นข้าวเที่ยง ช่วงบ่ายๆก็ไป สอยมะพร้าวแก่มาทำวุ้นมะพร้าว ตอนรอวุ้นเซ็ตตัวก็เลยนอนพัก ตื่นอีกทีก็ตอนเย็น มาย่างคอหมูย่างกิรฃนกับครอบครัว วุ้นก็รสชาติอร่อยแต่เหมือตอนต้มวุ้นจะกะไฟไม่ดี วุ้นเลยเหลวๆหน่อย แล้วก็กินก๋วยเตี๋ยวอีกชามนึงเป็นอาหารเย็น'